In [ ]:
!python -m pip install -e ..

In [31]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from torch_measure.datasets import load
from torch_measure.models import Rasch, TwoPL, ThreePL
from torch_measure.data import ResponseMatrix
from torch_measure.viz import plot_icc, plot_response_heatmap, plot_information
import torch_measure
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

import warnings
warnings.filterwarnings("ignore")

Using device: cpu


### Load the solver response matrix

Rows are solver models, columns are MMLU-Pro items, and values are binary correctness. Missing or unparsed responses stay as `NaN` and are masked by the fitting code.

In [32]:
matrix_path = "../benchmarks/mmlu/response_matrices/mmlu_pro_solver_response_matrix.csv"
subject_meta_path = "../benchmarks/mmlu/response_matrices/mmlu_pro_solver_subject_metadata.csv"
item_meta_path = "../benchmarks/mmlu/response_matrices/mmlu_pro_solver_item_metadata.csv"

df = pd.read_csv(matrix_path, index_col=0)
subject_meta = pd.read_csv(subject_meta_path)
item_meta = pd.read_csv(item_meta_path)

rm = ResponseMatrix(
    data=torch.tensor(df.values, dtype=torch.float32),
    subject_ids=list(df.index.astype(str)),
    item_ids=list(df.columns.astype(str)),
    item_contents=list(item_meta["source"].astype(str)),
    subject_metadata=subject_meta.to_dict("records"),
    info={
        "benchmark_id": "mmlu_pro_solver",
        "item_id_field": "original_id",
        "value": "correct: 1.0=true, 0.0=false, NaN=unparsed",
    },
)

print(f"{rm.n_subjects} models x {rm.n_items} items, density = {rm.density:.1%}")

10 models x 154 items, density = 73.6%


### Regular IRT fits

These fits keep the usual orientation: models are subjects and MMLU-Pro questions are items. The subject `ability` parameters are therefore the direct model capability estimates.

In [33]:
def center_item_difficulty(model):
    with torch.no_grad():
        shift = model.difficulty.mean()
        model.difficulty.sub_(shift)
        model.ability.sub_(shift)
    return model


regular_fits = {}

regular_specs = {
    "1pl_regular": (Rasch, {"max_epochs": 10000, "lr": 0.01}),
    "2pl_regular": (TwoPL, {"max_epochs": 10000, "lr": 0.005}),
    "3pl_regular": (ThreePL, {"max_epochs": 10000, "lr": 0.003}),
}

for fit_name, (model_cls, fit_kwargs) in regular_specs.items():
    print(f"\nFitting {fit_name}")
    model = model_cls(rm.n_subjects, rm.n_items, device=device)
    history = model.fit(
        rm.data,
        method="mle",
        verbose=True,
        **fit_kwargs,
    )
    center_item_difficulty(model)
    regular_fits[fit_name] = {"model": model, "history": history}
    print(f"{fit_name} final loss: {history['losses'][-1]:.4f}")


Fitting 1pl_regular


MLE fitting:  22%|██▏       | 2154/10000 [00:05<00:19, 408.97it/s, loss=0.360577]


1pl_regular final loss: 0.3606

Fitting 2pl_regular


MLE fitting:  24%|██▍       | 2383/10000 [00:06<00:20, 369.45it/s, loss=0.228364]


2pl_regular final loss: 0.2284

Fitting 3pl_regular


MLE fitting:  27%|██▋       | 2665/10000 [00:08<00:22, 330.00it/s, loss=0.245170]

3pl_regular final loss: 0.2452


### 1PL item-marginal MMLE

The library's built-in EM marginalizes over abilities. For model capability, this custom 1PL fit instead integrates out item difficulties and estimates only model abilities.

In [46]:
rasch_item_marginal = Rasch(
    n_subjects=rm.n_items,      # original items/tasks
    n_items=rm.n_subjects,      # original models
    device=device,
)

hist_1pl_item_marginal = rasch_item_marginal.fit(
    rm.data.T,
    method="em",
    max_epochs=500,
    lr=0.01,
    n_quadrature=31,
    latent_prior_sd =1.5,
    verbose=True,
)

# After flipping, each original model is an item. Lower fitted difficulty means
# stronger model capability, so capability is negative difficulty.
theta_1pl_item_marginal = -rasch_item_marginal.difficulty.detach().cpu()
theta_1pl_item_marginal = theta_1pl_item_marginal - theta_1pl_item_marginal.mean()

print(f"1PL item-marginal final loss: {hist_1pl_item_marginal['losses_ability'][-1]:.4f}")

EM: abilities: 100%|██████████| 500/500 [00:02<00:00, 176.45it/s, loss=0.399066]


1PL item-marginal final loss: 0.3991


### 2PL and 3PL item-marginalized IRT

In [ ]:
import math
import torch.nn.functional as F
from tqdm.auto import tqdm

# a, c <-- random or still fitting?

def fit_abilities_item_marginalized_pl(
    data,
    pl=2,
    n_samples=1024,
    max_epochs=1000,
    lr=0.03,
    b_sd=1.5,
    log_a_mean=0.0,
    log_a_sd=0.5,
    c_alpha=1.0,
    c_beta=9.0,
    ability_prior_sd=3.0,
    device="cpu",
    seed=0,
):
    assert pl in {2, 3}

    Y = data.to(device).float()
    mask = ~torch.isnan(Y) & (Y != -1)

    n_models, n_items = Y.shape
    ability = torch.nn.Parameter(torch.zeros(n_models, device=device))

    gen = torch.Generator(device=device)
    gen.manual_seed(seed)

    b_samples = torch.randn(n_samples, generator=gen, device=device) * b_sd

    log_a_samples = (
        torch.randn(n_samples, generator=gen, device=device) * log_a_sd
        + log_a_mean
    )
    a_samples = torch.exp(log_a_samples)

    if pl == 3:
        beta = torch.distributions.Beta(
            torch.tensor(c_alpha, device=device),
            torch.tensor(c_beta, device=device),
        )
        c_samples = beta.sample((n_samples,))
    else:
        c_samples = torch.zeros(n_samples, device=device)

    log_weight = -math.log(n_samples)
    n_obs = mask.sum().clamp_min(1).float()

    opt = torch.optim.Adam([ability], lr=lr)
    losses = []

    for _ in tqdm(range(max_epochs), desc=f"{pl}PL item-marginal MMLE"):
        opt.zero_grad()
        item_terms = []

        for item_idx in range(n_items):
            obs = mask[:, item_idx]
            if not obs.any():
                continue

            y = Y[obs, item_idx]
            theta = ability[obs]

            logits = a_samples[None, :] * (
                theta[:, None] - b_samples[None, :]
            )

            if pl == 2:
                logp = (
                    y[:, None] * F.logsigmoid(logits)
                    + (1 - y[:, None]) * F.logsigmoid(-logits)
                )
            else:
                p = c_samples[None, :] + (1 - c_samples[None, :]) * torch.sigmoid(logits)
                p = p.clamp(1e-7, 1 - 1e-7)
                logp = y[:, None] * torch.log(p) + (1 - y[:, None]) * torch.log1p(-p)

            item_terms.append(torch.logsumexp(log_weight + logp.sum(dim=0), dim=0))

        log_likelihood = torch.stack(item_terms).sum()
        prior = 0.5 * (ability / ability_prior_sd).pow(2).mean()
        loss = -log_likelihood / n_obs + prior

        loss.backward()
        opt.step()
        losses.append(loss.item())

    return ability.detach(), losses

In [48]:
theta_2pl_item_marginal, loss_2pl_item_marginal = fit_abilities_item_marginalized_pl(
    rm.data,
    pl=2,
    n_samples=2048,
    max_epochs=1000,
    lr=0.03,
    b_sd=1.5,
    device=device,
    seed=0,
)

theta_2pl_item_marginal = theta_2pl_item_marginal - theta_2pl_item_marginal.mean()

2PL item-marginal MMLE:   0%|          | 0/1000 [00:00<?, ?it/s]

In [49]:
theta_3pl_item_marginal, loss_3pl_item_marginal = fit_abilities_item_marginalized_pl(
    rm.data,
    pl=3,
    n_samples=4096,
    max_epochs=1000,
    lr=0.03,
    b_sd=1.5,
    c_alpha=1.0,
    c_beta=9.0,
    device=device,
    seed=0,
)

theta_3pl_item_marginal = theta_3pl_item_marginal - theta_3pl_item_marginal.mean()

3PL item-marginal MMLE:   0%|          | 0/1000 [00:00<?, ?it/s]

### Final capability table

Accuracy is included as a sanity-check reference. The final table now includes both regular `1pl_regular` and item-marginalized `1pl_item_marginal_mmle` capability estimates.

In [50]:
accuracy = df.mean(axis=1, skipna=True)
n_observed = df.notna().sum(axis=1)

capability_df = pd.DataFrame({
    "model": rm.subject_ids,
    "accuracy": accuracy.loc[rm.subject_ids].values,
    "n_observed": n_observed.loc[rm.subject_ids].values,
    "1pl_regular": regular_fits["1pl_regular"]["model"].ability.detach().cpu().numpy(),
    "1pl_item_marginal_mmle": theta_1pl_item_marginal.cpu().numpy(),
    "2pl_regular": regular_fits["2pl_regular"]["model"].ability.detach().cpu().numpy(),
    "2pl_item_marginal_mmle": theta_2pl_item_marginal.cpu().numpy(),
    "3pl_regular": regular_fits["3pl_regular"]["model"].ability.detach().cpu().numpy(),
    "3pl_item_marginal_mmle": theta_3pl_item_marginal.cpu().numpy(),
})

ability_cols = [
    "1pl_regular",
    "1pl_item_marginal_mmle",
    "2pl_regular",
    "2pl_item_marginal_mmle",
    "3pl_regular",
    "3pl_item_marginal_mmle"
]

for col in ability_cols:
    capability_df[f"{col}_rank"] = capability_df[col].rank(
        ascending=False,
        method="min",
    ).astype(int)

capability_df = capability_df.sort_values("1pl_regular", ascending=False)
capability_df

,model,accuracy,n_observed,1pl_regular,1pl_item_marginal_mmle,2pl_regular,2pl_item_marginal_mmle,3pl_regular,3pl_item_marginal_mmle,1pl_regular_rank,1pl_item_marginal_mmle_rank,2pl_regular_rank,2pl_item_marginal_mmle_rank,3pl_regular_rank,3pl_item_marginal_mmle_rank
0,mistralai_ministral_3_14b_instruct_2512_bf16,0.406015,133,-1.290757,0.866439,-0.673109,0.626578,-1.022165,0.582651,1,1,2,1,4,1
2,mistralai_ministral_3_8b_instruct_2512_bf16,0.331081,148,-1.736354,0.543603,-0.656015,0.344329,-2.236598,0.298951,2,2,1,2,9,2
5,qwen_qwen3_5_2b,0.309735,113,-2.003569,0.445519,-1.463806,0.245722,-1.003447,0.235918,3,3,3,3,3,3
1,mistralai_ministral_3_3b_instruct_2512_bf16,0.274648,142,-2.183990,0.275735,-1.666650,0.133408,-2.260408,0.098549,4,4,6,4,10,4
3,qwen_qwen3_5_0_8b,0.229167,96,-2.465466,0.033880,-1.540012,0.058652,-1.229515,0.051759,5,5,4,5,5,5
9,claude_sonnet_4_6,0.224299,107,-2.523444,0.006115,-2.274055,-0.049952,-0.943591,-0.069371,6,7,7,7,1,7
8,claude_haiku_4_5_20251001,0.228261,92,-2.666971,0.028861,-2.336909,-0.022239,-0.991918,-0.024386,7,6,9,6,2,6
4,qwen_qwen3_5_27b,0.120000,100,-3.462860,-0.745528,-3.023268,-0.427747,-2.220126,-0.369949,8,9,10,8,6,8
6,qwen_qwen3_5_4b,0.118812,101,-3.514381,-0.697797,-2.293454,-0.460449,-2.220346,-0.412690,9,8,8,10,7,10
7,qwen_qwen3_5_9b,0.118812,101,-3.526037,-0.756827,-1.649544,-0.448303,-2.220530,-0.391432,10,10,5,9,8,9


### Uncertainty quantification

For the regular IRT fits, use a diagonal Laplace/Fisher approximation for ability standard errors. For item-marginal fits, bootstrap over item columns because the item parameters have been integrated out.

In [55]:
def ability_laplace_se(model, data):
    data = data.to(model.device).float()
    mask = ~torch.isnan(data) & (data != -1)

    with torch.no_grad():
        theta = model.ability.detach()
        beta = model.difficulty.detach()

        if hasattr(model, "discrimination"):
            a = model.discrimination.detach()
        else:
            a = torch.ones_like(beta)

        logits = a.unsqueeze(0) * (theta.unsqueeze(1) - beta.unsqueeze(0))
        base_prob = torch.sigmoid(logits)

        if hasattr(model, "guessing"):
            c = model.guessing.detach()
            prob = c.unsqueeze(0) + (1 - c.unsqueeze(0)) * base_prob
            dprob_dtheta = (1 - c.unsqueeze(0)) * a.unsqueeze(0) * base_prob * (1 - base_prob)
        else:
            prob = base_prob
            dprob_dtheta = a.unsqueeze(0) * base_prob * (1 - base_prob)

        prob = prob.clamp(1e-7, 1 - 1e-7)
        info_matrix = dprob_dtheta.pow(2) / (prob * (1 - prob))
        info_matrix = torch.where(mask, info_matrix, torch.zeros_like(info_matrix))
        info = info_matrix.sum(dim=1)
        se = 1 / torch.sqrt(info.clamp_min(1e-8))

    return se.detach().cpu()


for fit_name in ["1pl_regular", "2pl_regular", "3pl_regular"]:
    se = ability_laplace_se(regular_fits[fit_name]["model"], rm.data)
    capability_df[f"{fit_name}_se"] = se.numpy()
    capability_df[f"{fit_name}_lo"] = capability_df[fit_name] - 1.96 * capability_df[f"{fit_name}_se"]
    capability_df[f"{fit_name}_hi"] = capability_df[fit_name] + 1.96 * capability_df[f"{fit_name}_se"]

capability_df[
    ["model", "accuracy", "n_observed"]
    + [
        "1pl_regular", "1pl_regular_se", "1pl_regular_lo", "1pl_regular_hi",
        "2pl_regular", "2pl_regular_se", "2pl_regular_lo", "2pl_regular_hi",
        "3pl_regular", "3pl_regular_se", "3pl_regular_lo", "3pl_regular_hi",
    ]
]

,model,accuracy,n_observed,1pl_regular,1pl_regular_se,1pl_regular_lo,1pl_regular_hi,2pl_regular,2pl_regular_se,2pl_regular_lo,2pl_regular_hi,3pl_regular,3pl_regular_se,3pl_regular_lo,3pl_regular_hi
0,mistralai_ministral_3_14b_instruct_2512_bf16,0.406015,133,-1.290757,0.228792,-1.739190,-0.842324,-0.673109,0.018675,-0.709711,-0.636507,-1.022165,0.015002,-1.051570,-0.992761
2,mistralai_ministral_3_8b_instruct_2512_bf16,0.331081,148,-1.736354,0.237535,-2.201923,-1.270785,-0.656015,0.015350,-0.686101,-0.625929,-2.236598,0.057042,-2.348399,-2.124796
5,qwen_qwen3_5_2b,0.309735,113,-2.003569,0.225395,-2.445343,-1.561796,-1.463806,0.018567,-1.500198,-1.427415,-1.003447,0.020500,-1.043628,-0.963267
1,mistralai_ministral_3_3b_instruct_2512_bf16,0.274648,142,-2.183990,0.285258,-2.743095,-1.624884,-1.666650,0.057184,-1.778731,-1.554569,-2.260408,0.121727,-2.498992,-2.021824
3,qwen_qwen3_5_0_8b,0.229167,96,-2.465466,0.339483,-3.130853,-1.800079,-1.540012,0.539460,-2.597353,-0.482670,-1.229515,0.028441,-1.285259,-1.173771
9,claude_sonnet_4_6,0.224299,107,-2.523444,0.249289,-3.012052,-2.034837,-2.274055,0.053454,-2.378824,-2.169285,-0.943591,0.007325,-0.957948,-0.929233
8,claude_haiku_4_5_20251001,0.228261,92,-2.666971,0.340042,-3.333453,-2.000489,-2.336909,0.033586,-2.402738,-2.271080,-0.991918,0.026592,-1.044039,-0.939797
4,qwen_qwen3_5_27b,0.120000,100,-3.462860,0.340279,-4.129806,-2.795914,-3.023268,0.022143,-3.066668,-2.979867,-2.220126,0.025143,-2.269407,-2.170846
6,qwen_qwen3_5_4b,0.118812,101,-3.514381,0.289004,-4.080829,-2.947932,-2.293454,0.032260,-2.356684,-2.230223,-2.220346,0.011038,-2.241981,-2.198711
7,qwen_qwen3_5_9b,0.118812,101,-3.526037,0.272646,-4.060423,-2.991652,-1.649544,0.099244,-1.844062,-1.455025,-2.220530,0.026863,-2.273181,-2.167878


In [56]:
def fit_1pl_item_marginal_capability(data, max_epochs=200, lr=0.01, n_quadrature=31, device="cpu", seed=0):
    model = Rasch(
        n_subjects=data.shape[1],
        n_items=data.shape[0],
        device=device,
    )
    _ = model.fit(
        data.T,
        method="em",
        max_epochs=max_epochs,
        lr=lr,
        n_quadrature=n_quadrature,
        verbose=False,
    )
    theta = -model.difficulty.detach().cpu()
    return theta - theta.mean()


def bootstrap_item_capabilities(data, fit_fn, fit_kwargs, n_boot=50, seed=0):
    rng = np.random.default_rng(seed)
    n_models, n_items = data.shape
    boot = []

    for boot_idx in tqdm(range(n_boot), desc="Bootstrap items"):
        cols = rng.integers(0, n_items, size=n_items)
        data_b = data[:, cols]
        theta_b = fit_fn(data_b, **fit_kwargs, seed=seed + boot_idx)
        theta_b = theta_b.detach().cpu()
        theta_b = theta_b - theta_b.mean()
        boot.append(theta_b.numpy())

    return np.vstack(boot)


def add_bootstrap_summary(capability_df, boot, subject_ids, col):
    summary = pd.DataFrame({
        "model": subject_ids,
        f"{col}_boot_mean": boot.mean(axis=0),
        f"{col}_boot_se": boot.std(axis=0, ddof=1),
        f"{col}_boot_lo": np.percentile(boot, 2.5, axis=0),
        f"{col}_boot_hi": np.percentile(boot, 97.5, axis=0),
    })
    return capability_df.merge(summary, on="model", how="left")


# Set to True when you want bootstrap SEs for item-marginal capabilities.
RUN_ITEM_BOOTSTRAP = False
N_BOOT = 50

if RUN_ITEM_BOOTSTRAP:
    boot_1pl = bootstrap_item_capabilities(
        rm.data,
        fit_1pl_item_marginal_capability,
        fit_kwargs={"max_epochs": 200, "lr": 0.01, "n_quadrature": 31, "device": device},
        n_boot=N_BOOT,
        seed=101,
    )
    capability_df = add_bootstrap_summary(capability_df, boot_1pl, rm.subject_ids, "1pl_item_marginal_mmle")

    boot_2pl = bootstrap_item_capabilities(
        rm.data,
        fit_abilities_item_marginalized_pl,
        fit_kwargs={"pl": 2, "n_samples": 1024, "max_epochs": 300, "lr": 0.03, "b_sd": 1.5, "device": device},
        n_boot=N_BOOT,
        seed=202,
    )
    capability_df = add_bootstrap_summary(capability_df, boot_2pl, rm.subject_ids, "2pl_item_marginal_mmle")

    boot_3pl = bootstrap_item_capabilities(
        rm.data,
        fit_abilities_item_marginalized_pl,
        fit_kwargs={"pl": 3, "n_samples": 2048, "max_epochs": 300, "lr": 0.03, "b_sd": 1.5, "c_alpha": 1.0, "c_beta": 9.0, "device": device},
        n_boot=N_BOOT,
        seed=303,
    )
    capability_df = add_bootstrap_summary(capability_df, boot_3pl, rm.subject_ids, "3pl_item_marginal_mmle")

capability_df

,model,accuracy,n_observed,1pl_regular,1pl_item_marginal_mmle,2pl_regular,2pl_item_marginal_mmle,3pl_regular,3pl_item_marginal_mmle,1pl_regular_rank,...,3pl_item_marginal_mmle_rank,1pl_regular_se,2pl_regular_se,3pl_regular_se,1pl_regular_lo,1pl_regular_hi,2pl_regular_lo,2pl_regular_hi,3pl_regular_lo,3pl_regular_hi
0,mistralai_ministral_3_14b_instruct_2512_bf16,0.406015,133,-1.290757,0.866439,-0.673109,0.626578,-1.022165,0.582651,1,...,1,0.228792,0.018675,0.015002,-1.739190,-0.842324,-0.709711,-0.636507,-1.051570,-0.992761
2,mistralai_ministral_3_8b_instruct_2512_bf16,0.331081,148,-1.736354,0.543603,-0.656015,0.344329,-2.236598,0.298951,2,...,2,0.237535,0.015350,0.057042,-2.201923,-1.270785,-0.686101,-0.625929,-2.348399,-2.124796
5,qwen_qwen3_5_2b,0.309735,113,-2.003569,0.445519,-1.463806,0.245722,-1.003447,0.235918,3,...,3,0.225395,0.018567,0.020500,-2.445343,-1.561796,-1.500198,-1.427415,-1.043628,-0.963267
1,mistralai_ministral_3_3b_instruct_2512_bf16,0.274648,142,-2.183990,0.275735,-1.666650,0.133408,-2.260408,0.098549,4,...,4,0.285258,0.057184,0.121727,-2.743095,-1.624884,-1.778731,-1.554569,-2.498992,-2.021824
3,qwen_qwen3_5_0_8b,0.229167,96,-2.465466,0.033880,-1.540012,0.058652,-1.229515,0.051759,5,...,5,0.339483,0.539460,0.028441,-3.130853,-1.800079,-2.597353,-0.482670,-1.285259,-1.173771
9,claude_sonnet_4_6,0.224299,107,-2.523444,0.006115,-2.274055,-0.049952,-0.943591,-0.069371,6,...,7,0.249289,0.053454,0.007325,-3.012052,-2.034837,-2.378824,-2.169285,-0.957948,-0.929233
8,claude_haiku_4_5_20251001,0.228261,92,-2.666971,0.028861,-2.336909,-0.022239,-0.991918,-0.024386,7,...,6,0.340042,0.033586,0.026592,-3.333453,-2.000489,-2.402738,-2.271080,-1.044039,-0.939797
4,qwen_qwen3_5_27b,0.120000,100,-3.462860,-0.745528,-3.023268,-0.427747,-2.220126,-0.369949,8,...,8,0.340279,0.022143,0.025143,-4.129806,-2.795914,-3.066668,-2.979867,-2.269407,-2.170846
6,qwen_qwen3_5_4b,0.118812,101,-3.514381,-0.697797,-2.293454,-0.460449,-2.220346,-0.412690,9,...,10,0.289004,0.032260,0.011038,-4.080829,-2.947932,-2.356684,-2.230223,-2.241981,-2.198711
7,qwen_qwen3_5_9b,0.118812,101,-3.526037,-0.756827,-1.649544,-0.448303,-2.220530,-0.391432,10,...,9,0.272646,0.099244,0.026863,-4.060423,-2.991652,-1.844062,-1.455025,-2.273181,-2.167878


### AUC Evaluation - Model Selection

Use an 80:20 split over observed cells. Each candidate IRT model is refit on the training cells, predicts the held-out cells, and is evaluated by held-out AUC. The regular 1PL/2PL/3PL models make item-specific predictions; the item-marginal 2PL/3PL fits make model-level marginal predictions by integrating over the same item-parameter priors used during fitting.

In [68]:
from torch_measure.data import random_mask
from torch_measure.metrics.functional import compute_all
from torch_measure.models import predict_dense


def _heldout_metric_row(fit_name, rep, metrics, data, test_mask):
    y_test = data[test_mask].float()
    return {
        "fit": fit_name,
        "rep": rep,
        "n_test": int(test_mask.sum().item()),
        "test_positive_rate": y_test.mean().item(),
        "heldout_auc": metrics["auc"],
        "heldout_log_likelihood": metrics["log_likelihood"],
        "heldout_brier": metrics["brier"],
        "heldout_ece": metrics["ece"],
    }


def heldout_auc_for_regular_model(model_cls, data, fit_kwargs, train_mask, test_mask):
    model = model_cls(data.shape[0], data.shape[1], device=device)
    history = model.fit(
        data,
        mask=train_mask,
        method="mle",
        verbose=False,
        **fit_kwargs,
    )
    center_item_difficulty(model)

    probs = predict_dense(model).detach()
    metrics = compute_all(probs, data, mask=test_mask)
    return metrics, model, history


def heldout_auc_for_1pl_item_marginal(data, fit_kwargs, train_mask, test_mask):
    flipped = Rasch(
        n_subjects=data.shape[1],
        n_items=data.shape[0],
        device=device,
    )
    history = flipped.fit(
        data.T,
        mask=train_mask.T,
        method="em",
        verbose=False,
        **fit_kwargs,
    )
    center_item_difficulty(flipped)

    probs = predict_dense(flipped).detach().T
    metrics = compute_all(probs, data, mask=test_mask)
    return metrics, flipped, history


def item_marginal_prior_probs(
    theta,
    n_items,
    pl,
    n_samples=8192,
    b_sd=1.5,
    log_a_mean=0.0,
    log_a_sd=0.5,
    c_alpha=1.0,
    c_beta=9.0,
    seed=0,
):
    theta = theta.to(device).float()
    torch.manual_seed(seed)
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)

    b_samples = torch.randn(n_samples, generator=gen, device=device) * b_sd
    log_a_samples = torch.randn(n_samples, generator=gen, device=device) * log_a_sd + log_a_mean
    a_samples = torch.exp(log_a_samples)

    logits = a_samples[None, :] * (theta[:, None] - b_samples[None, :])
    if pl == 2:
        p_samples = torch.sigmoid(logits)
    elif pl == 3:
        beta = torch.distributions.Beta(
            torch.tensor(c_alpha, device=device),
            torch.tensor(c_beta, device=device),
        )
        c_samples = beta.sample((n_samples,))
        p_samples = c_samples[None, :] + (1 - c_samples[None, :]) * torch.sigmoid(logits)
    else:
        raise ValueError(f"Expected pl=2 or pl=3, got {pl!r}")

    p_model = p_samples.mean(dim=1)
    return p_model[:, None].expand(-1, n_items)


def heldout_auc_for_prior_item_marginal(data, fit_kwargs, predict_kwargs, train_mask, test_mask, seed=0):
    train_data = data.clone()
    train_data[~train_mask] = float("nan")

    theta, history = fit_abilities_item_marginalized_pl(
        train_data,
        **fit_kwargs,
        seed=seed,
    )
    theta = theta - theta.mean()

    probs = item_marginal_prior_probs(
        theta,
        n_items=data.shape[1],
        seed=seed + 100_000,
        **predict_kwargs,
    ).detach()
    metrics = compute_all(probs, data, mask=test_mask)
    return metrics, theta, history


item_marginal_specs = {
    "1pl_item_marginal_mmle": {
        "kind": "flipped_1pl",
        "fit_kwargs": {"max_epochs": 500, "lr": 0.01, "n_quadrature": 31},
    },
    "2pl_item_marginal_mmle": {
        "kind": "prior_marginal",
        "fit_kwargs": {"pl": 2, "n_samples": 2048, "max_epochs": 1000, "lr": 0.03, "b_sd": 1.5, "device": device},
        "predict_kwargs": {"pl": 2, "n_samples": 8192, "b_sd": 1.5},
    },
    "3pl_item_marginal_mmle": {
        "kind": "prior_marginal",
        "fit_kwargs": {"pl": 3, "n_samples": 4096, "max_epochs": 1000, "lr": 0.03, "b_sd": 1.5, "c_alpha": 1.0, "c_beta": 9.0, "device": device},
        "predict_kwargs": {"pl": 3, "n_samples": 8192, "b_sd": 1.5, "c_alpha": 1.0, "c_beta": 9.0},
    },
}


def evaluate_heldout_auc(regular_specs, item_marginal_specs, data, n_repeats=1, train_frac=0.8, seed=0):
    data = data.to(device).float()
    observed = ~torch.isnan(data) & (data != -1)
    rows = []

    for rep in range(n_repeats):
        torch.manual_seed(seed + rep)
        train_mask, test_mask = random_mask(observed, train_frac=train_frac)

        for fit_name, (model_cls, fit_kwargs) in regular_specs.items():
            metrics, _, _ = heldout_auc_for_regular_model(
                model_cls,
                data,
                fit_kwargs,
                train_mask=train_mask,
                test_mask=test_mask,
            )
            rows.append(_heldout_metric_row(fit_name, rep, metrics, data, test_mask))

        for fit_name, spec in item_marginal_specs.items():
            if spec["kind"] == "flipped_1pl":
                metrics, _, _ = heldout_auc_for_1pl_item_marginal(
                    data,
                    spec["fit_kwargs"],
                    train_mask=train_mask,
                    test_mask=test_mask,
                )
            elif spec["kind"] == "prior_marginal":
                metrics, _, _ = heldout_auc_for_prior_item_marginal(
                    data,
                    spec["fit_kwargs"],
                    spec["predict_kwargs"],
                    train_mask=train_mask,
                    test_mask=test_mask,
                    seed=seed + rep,
                )
            else:
                raise ValueError(f"Unknown item-marginal spec kind: {spec['kind']!r}")

            rows.append(_heldout_metric_row(fit_name, rep, metrics, data, test_mask))

    return pd.DataFrame(rows)

In [ ]:
heldout_eval_raw = evaluate_heldout_auc(
    regular_specs,
    item_marginal_specs,
    rm.data,
    n_repeats=1,
    train_frac=0.8,
    seed=123,
)

heldout_eval_summary = (
    heldout_eval_raw
    .groupby("fit")
    .agg(
        n_test_mean=("n_test", "mean"),
        test_positive_rate_mean=("test_positive_rate", "mean"),
        heldout_auc_mean=("heldout_auc", "mean"),
        heldout_auc_sd=("heldout_auc", "std"),
        heldout_log_likelihood_mean=("heldout_log_likelihood", "mean"),
        heldout_log_likelihood_sd=("heldout_log_likelihood", "std"),
        heldout_brier_mean=("heldout_brier", "mean"),
        heldout_brier_sd=("heldout_brier", "std"),
        heldout_ece_mean=("heldout_ece", "mean"),
        heldout_ece_sd=("heldout_ece", "std"),
    )
    .reset_index()
    .sort_values("heldout_auc_mean", ascending=False)
)

heldout_eval_summary

2PL item-marginal MMLE:   0%|          | 0/1000 [00:00<?, ?it/s]

3PL item-marginal MMLE:   0%|          | 0/1000 [00:00<?, ?it/s]

2PL item-marginal MMLE:   0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 